# Entraînement distribué avec DDP

Dans ce TP, nous allons répartir l'entraînement d'un modèle sur plusieurs processus avec `DistributedDataParallel` (DDP), le mécanisme standard de PyTorch pour le **parallélisme de données**.

Colab ne fournit qu'un seul GPU : nous lancerons donc 2 processus **sur CPU**, qui communiquent avec le backend `gloo`. Le code est exactement celui qu'on utiliserait sur plusieurs GPUs, au backend près (`nccl`) : ce que vous écrirez ici fonctionne tel quel sur une machine à 4 ou 8 GPUs.

Au programme :

1. écrire un script d'entraînement DDP et le lancer avec `torchrun` ;
2. vérifier que les processus restent synchronisés ;
3. sauvegarder un checkpoint correctement ;
4. faire la même chose en quelques lignes avec PyTorch Lightning.

In [ ]:
!pip install -q lightning

## Le problème de départ

Voici un script d'entraînement classique, sur un seul processus, pour un problème de classification synthétique. Le cadre `%%writefile` écrit le contenu de la cellule dans un fichier au lieu de l'exécuter.

In [ ]:
%%writefile common.py
import torch
from torch import nn
from torch.utils.data import TensorDataset


def make_dataset(n: int = 8192) -> TensorDataset:
  # Même graine dans tous les processus : tous voient le même jeu de données
  generator = torch.Generator().manual_seed(0)
  x = torch.randn(n, 20, generator=generator)
  true_weights = torch.randn(20, 3, generator=generator)
  y = (x @ true_weights).argmax(dim=1)
  return TensorDataset(x, y)


def make_model() -> nn.Module:
  torch.manual_seed(0)
  return nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 3))

In [ ]:
%%writefile train_single.py
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model

dataset = make_dataset()
model = make_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

for epoch in range(3):
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  print(f"Epoch {epoch} : {len(loader)} batchs, dernière perte {loss.item():.3f}")

In [ ]:
!python train_single.py

## Un script DDP

Avec DDP, `torchrun` lance plusieurs **processus** qui exécutent le même script. Chaque processus :

- rejoint le groupe de processus (`dist.init_process_group`) ; `torchrun` lui fournit son numéro (*rang*) et le nombre total de processus dans des variables d'environnement ;
- enveloppe son modèle dans `DistributedDataParallel`, qui moyenne les gradients entre processus pendant `backward()` ;
- ne lit qu'une part du jeu de données, grâce à un `DistributedSampler`.

*Complétez le script `train_ddp.py` ci-dessous (les `...`) :*

- *initialisez le groupe de processus avec le backend `"gloo"` et récupérez le rang et le nombre de processus (`dist.get_rank()`, `dist.get_world_size()`) ;*
- *enveloppez le modèle dans `DDP` ;*
- *créez un `DistributedSampler` et passez-le au `DataLoader` (à la place de `shuffle=True`) ;*
- *appelez `sampler.set_epoch(epoch)` au début de chaque epoch ;*
- *n'affichez les messages que depuis le processus de rang 0.*

In [ ]:
%%writefile train_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

...  # Initialisation du groupe de processus, rang et nombre de processus

dataset = make_dataset()
model = ...  # Modèle enveloppé dans DDP
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = ...
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  ...
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  ...  # Affichage depuis le rang 0 seulement

dist.destroy_process_group()

### Solution

In [ ]:
%%writefile train_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

dist.init_process_group("gloo")  # "nccl" sur GPU
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model())  # sur GPU : DDP(make_model().to(rank), device_ids=[rank])
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  if rank == 0:
    print(f"Epoch {epoch} : {len(loader)} batchs par processus, "
          f"{world_size} processus, dernière perte {loss.item():.3f}")

dist.destroy_process_group()

Lancez le script sur 2 processus :

In [ ]:
!torchrun --nproc_per_node=2 train_ddp.py

*Répondez aux questions suivantes :*

1. *Combien de batchs chaque processus traite-t-il par epoch, comparé au script sur un seul processus ? Pourquoi ?*
2. *Chaque processus utilise des batchs de 64 exemples. Combien d'exemples contribuent à chaque mise à jour des poids ? Quelle conséquence sur le choix du learning rate ?*
3. *À quoi sert `sampler.set_epoch(epoch)` ?*

*Vos réponses ici.*

### Solution

1. Chaque processus traite 64 batchs par epoch au lieu de 128 : le `DistributedSampler` donne à chaque processus une moitié différente du jeu de données. Avec 2 GPUs, une epoch prend donc environ deux fois moins de temps.
2. Les gradients sont moyennés entre les 2 processus : chaque mise à jour utilise 2 × 64 = 128 exemples. Le batch effectif grandit avec le nombre de processus ; on augmente souvent le learning rate en proportion (avec un *warmup*), sinon l'entraînement avance moins par epoch.
3. Le `DistributedSampler` mélange les données avec une graine qui dépend de l'epoch : sans `set_epoch`, chaque epoch verrait les exemples dans le même ordre.

## Les processus restent-ils synchronisés ?

Chaque processus a sa propre copie du modèle. DDP garantit qu'elles restent identiques : les poids sont diffusés depuis le rang 0 à la création du `DDP`, puis tous les processus appliquent la même mise à jour, calculée à partir des gradients moyennés.

*Vérifions-le. Dans une copie du script (`train_ddp_check.py`), après l'entraînement, calculez dans chaque processus la somme de tous les poids du modèle, puis rassemblez ces sommes avec `dist.all_gather` et affichez-les depuis le rang 0.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
%%writefile train_ddp_check.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

dist.init_process_group("gloo")
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model())
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    cross_entropy(model(x), y).backward()
    optimizer.step()

checksum = torch.stack([p.detach().sum() for p in model.parameters()]).sum()
checksums = [torch.zeros(()) for _ in range(world_size)]
dist.all_gather(checksums, checksum)
if rank == 0:
  print("Somme des poids dans chaque processus :", [f"{c.item():.6f}" for c in checksums])

dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 train_ddp_check.py

## Sauvegarder un checkpoint

Dans un entraînement distribué, tous les processus ont les mêmes poids : un seul doit écrire le checkpoint, sinon ils écrivent tous le même fichier en même temps.

*Modifiez `train_ddp.py` pour sauvegarder le modèle à la fin de l'entraînement, depuis le rang 0 seulement, dans `model.pt`. Vérifiez ensuite, dans une cellule du notebook, que vous pouvez recharger ces poids dans un modèle **non distribué** créé avec `make_model()`.*

*Indice : le modèle d'origine est accessible dans `model.module`. Pourquoi faut-il sauvegarder `model.module.state_dict()` plutôt que `model.state_dict()` ?*

In [ ]:
# Votre code ici

### Solution

In [ ]:
%%writefile train_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

dist.init_process_group("gloo")
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model())
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  if rank == 0:
    print(f"Epoch {epoch} : dernière perte {loss.item():.3f}")

if rank == 0:
  torch.save(model.module.state_dict(), "model.pt")
  print("Checkpoint sauvegardé")

dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 train_ddp.py

In [ ]:
import torch

from common import make_model

reloaded = make_model()
reloaded.load_state_dict(torch.load("model.pt"))
print("Poids rechargés dans un modèle classique")

`model.state_dict()` préfixerait toutes les clés par `module.` (le nom de l'attribut dans lequel DDP range le modèle) : ces poids ne pourraient pas être rechargés dans un modèle ordinaire sans renommer les clés.

## Avec PyTorch Lightning

Lightning se charge de tout ce que nous avons écrit à la main : groupe de processus, `DDP`, `DistributedSampler`, `set_epoch`, affichage et sauvegarde depuis le rang 0. Il suffit de choisir une stratégie dans le `Trainer`.

*Complétez le `Trainer` de `train_lightning.py` pour entraîner sur 2 processus CPU avec la stratégie `"ddp"`, puis lancez le script. Sur une machine avec 4 GPUs, que faudrait-il changer ?*

In [ ]:
%%writefile train_lightning.py
import lightning as L
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model


class Classifier(L.LightningModule):
  def __init__(self) -> None:
    super().__init__()
    self.model = make_model()

  def training_step(self, batch, batch_idx):
    x, y = batch
    loss = cross_entropy(self.model(x), y)
    self.log("train_loss", loss, prog_bar=True)
    return loss

  def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=0.1)


if __name__ == "__main__":
  loader = DataLoader(make_dataset(), batch_size=64, shuffle=True)
  trainer = L.Trainer(max_epochs=3, ...)  # Votre code ici
  trainer.fit(Classifier(), loader)

### Solution

In [ ]:
%%writefile train_lightning.py
import lightning as L
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model


class Classifier(L.LightningModule):
  def __init__(self) -> None:
    super().__init__()
    self.model = make_model()

  def training_step(self, batch, batch_idx):
    x, y = batch
    loss = cross_entropy(self.model(x), y)
    self.log("train_loss", loss, prog_bar=True)
    return loss

  def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=0.1)


if __name__ == "__main__":
  loader = DataLoader(make_dataset(), batch_size=64, shuffle=True)
  trainer = L.Trainer(max_epochs=3, accelerator="cpu", devices=2, strategy="ddp",
                      enable_checkpointing=False, logger=False)
  trainer.fit(Classifier(), loader)

In [ ]:
!python train_lightning.py

Sur 4 GPUs, il suffirait d'écrire `accelerator="gpu", devices=4` (et d'ajouter `precision="bf16-mixed"` pour la précision mixte). Lightning remplace lui-même le `shuffle=True` du `DataLoader` par un `DistributedSampler`.

## Pour aller plus loin

- Si le modèle ne tient pas dans la mémoire d'un GPU, DDP ne suffit plus : la stratégie `"fsdp"` (*Fully Sharded Data Parallel*) répartit aussi les poids, les gradients et les états de l'optimiseur entre les GPUs.
- Sur plusieurs machines, `torchrun` prend en plus `--nnodes`, `--node_rank` et l'adresse d'une machine maître (`--rdzv_endpoint`).
- Voir le [tutoriel DDP de PyTorch](https://docs.pytorch.org/tutorials/intermediate/ddp_tutorial.html) et la [documentation de `torchrun`](https://docs.pytorch.org/docs/stable/elastic/run.html).